# Pre-entrega 4 · Parte 2 — Fine-tuning con LoRA en GPU
**Data Science III · Módulo 4** — Elizabeth Luque

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scarletluque0805/ds3-transformer-lora/blob/main/notebooks/04_parte2_finetuning_lora.ipynb)

**Paso 4 de la práctica:** entrenar, medir el tiempo y contar parámetros. Usa la configuración decidida en la Parte 1 (`reports/config_parte1.json`).

| Corrida | Modelo | r | α | learning rate | Para qué |
|---|---|---|---|---|---|
| **principal** | distilbert-base-uncased | 8 | 16 | 3e-4 | el modelo de la entrega |
| lr_3e-5 | distilbert-base-uncased | 8 | 16 | 3e-5 | demostrar por qué LoRA necesita un learning rate mayor que el fine-tuning completo |
| r4 / r16 | distilbert-base-uncased | 4 / 16 | 8 / 32 | 3e-4 | validar la elección de `r` |
| roberta (opcional) | roberta-base | 8 | 16 | 3e-4 | techo de referencia con el doble de cómputo |

En esta parte **solo se usa train y validation**. El test se evalúa una única vez en la Parte 3, contra el baseline del Módulo 3.

**Cómo correrlo:** `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU` y después `Entorno de ejecución → Ejecutar todo`. Tarda alrededor de 15 minutos. Al final se descarga `resultados_parte2.zip`. Después, `Archivo → Descargar → Descargar .ipynb`.

## 0. Entorno: repositorio, librerías y GPU

In [ ]:
import os, sys, subprocess
from pathlib import Path
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not Path("/content/ds3-transformer-lora").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/scarletluque0805/ds3-transformer-lora.git", "/content/ds3-transformer-lora"], check=True)
    os.chdir("/content/ds3-transformer-lora")
    # Colab trae torchao 0.10 y peft exige >= 0.16: como no lo usamos, se desinstala antes de importar peft
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "datasets", "accelerate"], check=True)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "data"))
print("Repositorio:", ROOT)

In [ ]:
import json, time, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import transformers, datasets
import peft
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

assert torch.cuda.is_available(), "Sin GPU: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU, y volver a ejecutar."
ENTORNO = {"gpu": torch.cuda.get_device_name(0), "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
           "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "datasets": datasets.__version__}
print(json.dumps(ENTORNO, indent=2))

## 1. Datos y tokenización (configuración de la Parte 1)

In [ ]:
from src import datos_hf as dh, modelos as mo, entrenamiento as en
CONFIG = json.loads((ROOT / "reports/config_parte1.json").read_text())
CKPT, MAX_LEN = CONFIG["modelo_base"], CONFIG["max_length"]
SEED = CONFIG["training_args"]["seed"]
tr, va, te, label2id, id2label = dh.preparar_dataframes()
ds = dh.a_datasetdict(tr, va, te)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CKPT)
ds_tok = en.tokenizar(ds, tok, MAX_LEN)
print(ds_tok)
print(f"Modelo base: {CKPT} | max_length: {MAX_LEN} | etiquetas: {label2id}")

## 2. Corrida principal: `distilbert-base-uncased` + LoRA (r=8, α=16, Q-V, lr=3e-4)
`TrainingArguments`: 4 épocas, batch de 32, `weight_decay=0.01`, warmup del 6%, `fp16` en la GPU, evaluación y checkpoint en cada época, y **se conserva la época con mejor F1-macro en validación** (`load_best_model_at_end`).

In [ ]:
OUT = ROOT / "reports" / "parte2"; (OUT / "figures").mkdir(parents=True, exist_ok=True)
lora = CONFIG["lora"]; ta = CONFIG["training_args"]
principal = en.entrenar("principal", CKPT, ds_tok, tok, id2label, label2id, r=lora["r"], lora_alpha=lora["lora_alpha"],
                        lora_dropout=lora["lora_dropout"], target_modules=lora["target_modules"], lr=ta["learning_rate"],
                        epochs=ta["num_train_epochs"], bs=ta["per_device_train_batch_size"], out_root="/content/resultados" if EN_COLAB else "resultados")
principal["modelo"].print_trainable_parameters()
print(f"\nTiempo de entrenamiento: {principal['segundos']:.1f} s ({principal['segundos'] / 60:.2f} min) en {ENTORNO['gpu']} | VRAM pico: {principal['vram_pico_gb']:.2f} GB")
print(f"Parámetros: {principal['params_entrenables']:,} entrenables de {principal['params_base']:,} ({principal['pct_entrenable']:.2f}%)")
print("\n--- ESTADÍSTICAS POR ÉPOCA (train vs. validación) ---")
print(principal["historial"].round(4).to_string(index=False))
principal["historial"].to_csv(OUT / "historial_principal.csv", index=False)

In [ ]:
h = principal["historial"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h.epoch, h.train_loss, "o-", lw=2, label="train loss")
axes[0].plot(h.epoch, h.val_loss, "s-", lw=2, color="#c55a11", label="validation loss")
axes[0].set(title="Pérdida (cross-entropy) por época", xlabel="época", ylabel="loss", xticks=h.epoch); axes[0].legend()
axes[1].plot(h.epoch, h.val_f1_macro, "s-", lw=2, color="#2f7d32", label="F1-macro validación")
axes[1].axhline(0.892, color="gray", ls=":", label="baseline Módulo 3 (F1 en test)")
axes[1].set(title="F1-macro en validación por época", xlabel="época", ylabel="F1-macro", xticks=h.epoch); axes[1].legend(loc="lower right")
fig.suptitle(f"{CKPT} + LoRA (r={lora['r']}, α={lora['lora_alpha']}) · {ENTORNO['gpu']} · {principal['segundos'] / 60:.1f} min", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / "figures/curvas_principal.png", dpi=150); plt.show()

In [ ]:
# Diagnóstico automático de la curva
gap = h.val_loss.iloc[-1] - h.train_loss.iloc[-1]
sube_val = bool((h.val_loss.diff().dropna() > 0.01).any())
print(f"Pérdida de train: {h.train_loss.iloc[0]:.4f} -> {h.train_loss.iloc[-1]:.4f} | validación: {h.val_loss.iloc[0]:.4f} -> {h.val_loss.iloc[-1]:.4f}")
print(f"Brecha final (val - train): {gap:+.4f}")
print(f"¿La pérdida de validación sube en alguna época? {'SÍ: posible inicio de sobreajuste' if sube_val else 'NO: sin señales de sobreajuste'}")
print(f"Mejor época por F1-macro de validación: {principal['mejor_epoca']} (F1 = {principal['val_f1_macro']:.4f})")

In [ ]:
# Guardar el adaptador LoRA (solo matrices A, B y el cabezal: pocos MB) para evaluarlo en la Parte 3
DIR_ADAPTER = ROOT / "models" / "lora_distilbert_r8"
principal["modelo"].save_pretrained(DIR_ADAPTER)
tam = sum(f.stat().st_size for f in DIR_ADAPTER.glob("*")) / 1e6
print(f"Adaptador guardado en {DIR_ADAPTER.relative_to(ROOT)}: {[f.name for f in DIR_ADAPTER.glob('*')]} ({tam:.1f} MB)")
print("Para comparar: el modelo completo pesa ~268 MB en fp32.")
resumen = [en.fila_resumen(principal)]
del principal["trainer"], principal["modelo"]; gc.collect(); torch.cuda.empty_cache()

## 3. Ablación del learning rate: 3e-5 (fine-tuning completo) vs. 3e-4 (LoRA)
Mismo modelo y misma configuración LoRA, cambiando solo el learning rate. En la clase, el error 3 decía "nunca 1e-3, usá 2e-5 o 5e-5", pero eso es para el fine-tuning completo. ¿Qué pasa con LoRA?

In [ ]:
corridas = {"principal": principal}
for nombre, kw in {"lr_3e-5": dict(r=8, lr=3e-5)}.items():
    res = en.entrenar(nombre, CKPT, ds_tok, tok, id2label, label2id, target_modules=lora["target_modules"], epochs=4, bs=32,
                      out_root="/content/resultados" if EN_COLAB else "resultados", **kw)
    res["historial"].to_csv(OUT / f"historial_{nombre}.csv", index=False)
    resumen.append(en.fila_resumen(res)); corridas[nombre] = res
    print(nombre, "->", resumen[-1])
    del res["trainer"], res["modelo"]; gc.collect(); torch.cuda.empty_cache()

## 4. Ablación del rango: r = 4 y r = 16 (α = 2r)

In [ ]:
for nombre, kw in {"r4": dict(r=4, lr=3e-4), "r16": dict(r=16, lr=3e-4)}.items():
    res = en.entrenar(nombre, CKPT, ds_tok, tok, id2label, label2id, target_modules=lora["target_modules"], epochs=4, bs=32,
                      out_root="/content/resultados" if EN_COLAB else "resultados", **kw)
    res["historial"].to_csv(OUT / f"historial_{nombre}.csv", index=False)
    resumen.append(en.fila_resumen(res)); corridas[nombre] = res
    print(nombre, "->", resumen[-1])
    del res["trainer"], res["modelo"]; gc.collect(); torch.cuda.empty_cache()

## 5. Techo de referencia (opcional): `roberta-base` con la misma configuración LoRA
Tiene el doble de capas. Sirve para medir cuánto se gana pagando el doble de cómputo. Se puede apagar con `EJECUTAR_ROBERTA = False`.

In [ ]:
EJECUTAR_ROBERTA = True
if EJECUTAR_ROBERTA:
    tok_rb = AutoTokenizer.from_pretrained("roberta-base")
    ds_tok_rb = en.tokenizar(ds, tok_rb, MAX_LEN)
    res = en.entrenar("roberta_r8", "roberta-base", ds_tok_rb, tok_rb, id2label, label2id, r=8, lr=3e-4, epochs=4, bs=32,
                      out_root="/content/resultados" if EN_COLAB else "resultados")
    res["historial"].to_csv(OUT / "historial_roberta_r8.csv", index=False)
    resumen.append(en.fila_resumen(res)); corridas["roberta_r8"] = res
    print("roberta_r8 ->", resumen[-1])
    del res["trainer"], res["modelo"]; gc.collect(); torch.cuda.empty_cache()

## 6. Resumen de corridas

In [ ]:
tabla = pd.DataFrame(resumen).set_index("corrida")
tabla.to_csv(OUT / "resumen_corridas.csv")
tabla

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for nombre, res in corridas.items():
    hh = res["historial"]
    axes[0].plot(hh.epoch, hh.val_loss, "o-", label=nombre)
    axes[1].plot(hh.epoch, hh.val_f1_macro, "o-", label=nombre)
axes[0].set(title="Pérdida de validación por corrida", xlabel="época", ylabel="val loss"); axes[0].legend()
axes[1].axhline(0.892, color="gray", ls=":", label="baseline M3 (test)")
axes[1].set(title="F1-macro de validación por corrida", xlabel="época", ylabel="F1-macro"); axes[1].legend(loc="lower right")
fig.tight_layout(); fig.savefig(OUT / "figures/ablacion.png", dpi=150); plt.show()

In [ ]:
p = tabla.loc["principal"]
filas = [
    ["La pérdida de train baja", "época final < época 1", f"{corridas['principal']['historial'].train_loss.iloc[0]:.3f} -> {p['train_loss final']:.3f}"],
    ["Sin sobreajuste marcado", "val loss sin subas > 0,01", "OK" if not sube_val else "revisar"],
    ["Parámetros entrenables", "< 3% del modelo", f"{p['% entrenable']}%"],
    ["lr 3e-5 aprende menos que 3e-4", "F1(3e-5) < F1(3e-4)", f"{tabla.loc['lr_3e-5', 'val F1-macro (mejor)']:.4f} vs {p['val F1-macro (mejor)']:.4f}"],
    ["r=8 vs r=4 / r=16", "diferencias chicas (tarea de bajo rango)", f"{tabla.loc['r4', 'val F1-macro (mejor)']:.4f} / {p['val F1-macro (mejor)']:.4f} / {tabla.loc['r16', 'val F1-macro (mejor)']:.4f}"],
    ["Entrena en minutos en una T4", "< 10 min", f"{p['minutos']} min"],
]
pd.DataFrame(filas, columns=["Prueba", "Esperado", "Observado"])

## 7. Guardar y descargar los resultados

In [ ]:
(OUT / "entorno.json").write_text(json.dumps(ENTORNO, indent=2))
import zipfile
zip_path = "/content/resultados_parte2.zip" if EN_COLAB else str(ROOT / "resultados_parte2.zip")
# Solo lo necesario: reports/parte2 (historiales, resumen, figuras, entorno) y el adaptador LoRA
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in [OUT, DIR_ADAPTER]:
        for f in carpeta.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(ROOT))
print("Generado:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
if EN_COLAB:
    from google.colab import files
    files.download(zip_path)